# ADVI for Taxi Trajectory Clustering
## Reproducing Section 4.4 of Kucukelbir et al. (2016)

**Reference:** *Automatic Differentiation Variational Inference*, Kucukelbir, A., Tran, D., Ranganath, R., Gelman, A., & Blei, D. M. (2016). arXiv:1603.00788.

---

### Objectives

This notebook implements the case study from Section 4.4 of the ADVI paper: **Exploring Millions of Taxi Trajectories** from the city of Porto.

The pipeline follows these steps:

1. **Data loading & preprocessing** — Load the Porto taxi dataset (1.7M rides, year 2014). Each trajectory is a variable-length sequence of (longitude, latitude) GPS coordinates recorded at 15-second intervals. We interpolate all trajectories to a fixed length of 50 coordinate pairs, yielding points in $\mathbb{R}^{100}$.

2. **Dimensionality reduction via PPCA with ARD** — Fit a Probabilistic PCA model with Automatic Relevance Determination on a subsample of 10 000 trajectories. PPCA-ARD automatically selects the effective number of principal components (the paper finds 11). Inference is performed using mean-field ADVI.

3. **Clustering via Gaussian Mixture Model** — Project all 1.7M trajectories into the learned subspace, then fit a GMM with $K=30$ components using ADVI with minibatch stochastic optimization.

4. **Visualization** — Display 50 000 randomly sampled trajectories colored by their cluster assignment on a map of Porto.

5. **Extension: Supervised PPCA (SUP-PPCA)** — Incorporate trip duration as a supervised signal into the dimensionality reduction step, then re-cluster. This reveals structure related to traffic patterns (e.g., inner vs. outer bridge routes).

### ADVI in a nutshell

For each model above, we use ADVI to approximate the posterior $p(\theta \mid x)$ with a Gaussian $q(\zeta; \phi) = \mathcal{N}(\zeta; \mu, \Sigma)$ in a transformed unconstrained space $\zeta = T(\theta)$. The algorithm:
- Transforms constrained latent variables to $\mathbb{R}^K$
- Uses the reparameterization trick (elliptical standardization) to express ELBO gradients as expectations over $\mathcal{N}(0, I)$
- Approximates gradients via Monte Carlo (single sample suffices)
- Optimizes with adaptive stochastic gradient ascent

## 1. Imports

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.interpolate import interp1d

torch.set_default_dtype(torch.float64)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"NumPy {np.__version__} | PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/Users/calla/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


NumPy 1.26.4 | PyTorch 2.6.0
CUDA available: False


## 2. Load the Porto Taxi Trajectories dataset

The dataset comes from the ECML-PKDD 2015 competition and contains **all 1.7 million taxi rides** in the city of Porto during the year 2014.

Each ride is described by:
- **POLYLINE**: a JSON-encoded list of `[longitude, latitude]` GPS coordinates sampled every 15 seconds.
- **TIMESTAMP**: Unix timestamp of the trip start.
- **TAXI_ID**: identifier of the taxi (442 taxis total).

We download it from Kaggle via `kagglehub`. If you haven't authenticated yet, run `kaggle configure` or set the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables.

In [2]:
import kagglehub

path = kagglehub.dataset_download("crailtap/taxi-trajectory")
print(f"Dataset downloaded to: {path}")

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
import os

csv_candidates = [
    os.path.join(path, "train.csv"),
    os.path.join(path, "train.csv.zip"),
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if csv_path is None:
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(".csv") or f.endswith(".csv.zip"):
                csv_path = os.path.join(root, f)
                break

print(f"Loading from: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print(f"\nNumber of unique taxis: {df['TAXI_ID'].nunique()}")
print(f"Columns: {list(df.columns)}")

### 2.1 Parse polylines and compute trip durations

Each `POLYLINE` is a JSON list of `[lon, lat]` pairs sampled every 15 s. The trip duration (in seconds) is `15 * (len(polyline) - 1)`. We discard trips flagged as having missing data or with fewer than 2 GPS points.

In [ ]:
df = df[df["MISSING_DATA"] == False].copy()

polylines = df["POLYLINE"].apply(json.loads)
lengths = polylines.apply(len)

# Drop trips with fewer than 2 GPS points (no trajectory to interpolate)
valid = lengths >= 2
polylines = polylines[valid]
df = df[valid].copy()
durations = (polylines.apply(len) - 1) * 15  # seconds

df["polyline"] = polylines
df["duration_s"] = durations.values
df["n_points"] = polylines.apply(len)

print(f"Valid trajectories: {len(df):,}")
print(f"Average trip duration: {df['duration_s'].mean():.0f} s  ({df['duration_s'].mean()/60:.1f} min)")
print(f"Average GPS points per trip: {df['n_points'].mean():.1f}")
print(f"Median GPS points per trip: {df['n_points'].median():.1f}")

### 2.2 Interpolate trajectories to a fixed length

Following the paper: *"The average trip is approximately 13 minutes long, which corresponds to 50 coordinates. We want to cluster independent of length, so we interpolate all trajectories to 50 coordinate pairs."*

This converts each trajectory into a point in $\mathbb{R}^{100}$ (50 longitudes + 50 latitudes, interleaved as `[lon_1, lat_1, ..., lon_50, lat_50]`).

In [ ]:
N_INTERP = 50  # fixed number of coordinate pairs per trajectory

def interpolate_trajectory(coords, n_points=N_INTERP):
    """Linearly interpolate a trajectory to exactly n_points (lon, lat) pairs."""
    coords = np.array(coords)
    n = len(coords)
    t_original = np.linspace(0, 1, n)
    t_new = np.linspace(0, 1, n_points)
    lon_interp = interp1d(t_original, coords[:, 0], kind="linear")(t_new)
    lat_interp = interp1d(t_original, coords[:, 1], kind="linear")(t_new)
    # Interleave: [lon1, lat1, lon2, lat2, ..., lon50, lat50]
    return np.column_stack([lon_interp, lat_interp]).ravel()

print(f"Interpolating {len(df):,} trajectories to {N_INTERP} coordinate pairs...")
X = np.vstack(df["polyline"].apply(interpolate_trajectory).values)
durations = df["duration_s"].values

print(f"Trajectory matrix X: shape {X.shape}  (N trajectories x {2*N_INTERP} features)")
print(f"Duration vector:     shape {durations.shape}")

### 2.3 Quick sanity check

Plot a few random trajectories to make sure the data looks reasonable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sample_idx = np.random.choice(len(X), size=3, replace=False)
for ax, idx in zip(axes, sample_idx):
    traj = X[idx].reshape(N_INTERP, 2)
    ax.plot(traj[:, 0], traj[:, 1], "o-", markersize=2, linewidth=0.8)
    ax.plot(traj[0, 0], traj[0, 1], "gs", markersize=8, label="start")
    ax.plot(traj[-1, 0], traj[-1, 1], "r^", markersize=8, label="end")
    ax.set_title(f"Trip {idx} — {durations[idx]/60:.1f} min")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=8)
    ax.set_aspect("equal")

plt.suptitle("Sample interpolated trajectories", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nDataset ready: {X.shape[0]:,} trajectories in R^{X.shape[1]}")
print(f"Duration range: {durations.min():.0f}s – {durations.max():.0f}s")